In [49]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotx
import pandas as pd
from pathlib import Path

plt.style.use(matplotx.styles.github['dark'])
DATA_DIR = Path("data")

In [50]:
Y = pd.read_csv(DATA_DIR / "small_movies_Y.csv", header=None)
R = pd.read_csv(DATA_DIR / "small_movies_R.csv", header=None)
movie_df = pd.read_csv(DATA_DIR / "small_movie_list.csv", index_col=0)
X = pd.read_csv(DATA_DIR / "small_movies_X.csv", header=None)
W = pd.read_csv(DATA_DIR / "small_movies_W.csv", header=None)
b = pd.read_csv(DATA_DIR / "small_movies_b.csv", header=None)

print("ratings_df:", Y.shape)
print("rated_df:", R.shape)
print("movie_df:", movie_df.shape)
print("movie_features_df:", X.shape)
print("user_features_df:", W.shape)
print("user_bias_df:", b.shape)

ratings_df: (4778, 443)
rated_df: (4778, 443)
movie_df: (4778, 3)
movie_features_df: (4778, 10)
user_features_df: (443, 10)
user_bias_df: (1, 443)


In [51]:
num_movies, num_features = X.shape
num_users, num_features = W.shape

print(num_movies, num_users, num_features)
X

4778 443 10


,0,1,2,3,4,5,6,7,8,9
0,-0.033287,1.166746,-0.506490,0.125096,1.534760,-0.210136,0.104176,-0.349303,0.753852,-0.208109
1,0.404058,2.266492,-0.742824,1.374039,1.145788,0.381959,-0.447495,-1.038103,-2.332266,-2.343191
2,0.023478,0.428102,-0.426381,0.107177,0.484193,-0.969298,-0.748563,-0.293784,0.439730,1.823422
3,0.511281,1.518797,1.602747,-2.296399,-0.570383,-0.480868,-0.662502,1.965215,0.495176,1.044845
4,-2.056514,0.458517,0.619371,-0.138456,0.011926,-0.282415,0.027502,-0.535465,0.661082,0.195084
...,...,...,...,...,...,...,...,...,...,...
4773,-0.473012,-1.069387,1.408808,-1.171842,1.222617,0.028973,0.230975,0.579943,-0.828117,-0.132676
4774,0.340389,3.040022,0.208837,-1.056347,-0.561943,0.583073,-1.171645,1.152431,0.363426,0.178877
4775,-0.438083,0.711810,-0.298449,-0.459011,1.824691,-0.036751,0.054586,0.148413,-0.439178,-0.062395
4776,1.137175,0.865196,1.066511,0.878567,0.579996,1.311802,-0.290936,0.619440,0.377445,-3.411507


In [53]:
movie1 = X.iloc[0]
movie1 = movie1[:2].to_numpy()
print(movie1)

user1 = W.iloc[0]
user1 = user1[:2].to_numpy()
print(user1)

print("Rating: ", np.dot(movie1, user1))

first_rated_value = Y.where(R == 1).stack().dropna()
first_rated_value

[-0.03328682  1.1667464 ]
[0.47935075 0.4766997 ]
Rating:  0.5402315943272112


0     112    5.0
      125    2.0
      297    4.0
      299    4.0
      327    2.0
            ... 
4773  127    3.5
4774  127    4.0
4775  127    3.5
4776  127    3.5
4777  127    3.5
Length: 39253, dtype: float64

In [54]:
# Find the first rated movie-user pair without reshaping the full ratings table
rated_positions = np.flatnonzero(R.to_numpy().ravel() == 1)
first_position = rated_positions[0]
movie_index, user_index = np.unravel_index(first_position, R.shape)

first_rated_value = Y.iat[movie_index, user_index]
movie_features = X.iloc[movie_index]
user_features = W.iloc[user_index]
user_bias = b.iat[0, user_index]
movie_details = movie_df.iloc[movie_index]
predicted_rating = np.dot(movie_features, user_features) + user_bias

print("Movie index:", movie_index)
print("User index:", user_index)
print("Actual rating:", first_rated_value)
print("Predicted rating:", predicted_rating)
print("User bias:", user_bias)
print("Movie details:")
display(movie_details)
print("Movie features from X:")
display(movie_features)
print("User features from W:")
display(user_features)

Movie index: 0
User index: 112
Actual rating: 5.0
Predicted rating: 0.4823925319108352
User bias: 0.29739505
Movie details:


mean rating                        3.4
number of ratings                    5
title                Yards, The (2000)
Name: 0, dtype: object

Movie features from X:


0   -0.033287
1    1.166746
2   -0.506490
3    0.125096
4    1.534760
5   -0.210136
6    0.104176
7   -0.349303
8    0.753852
9   -0.208109
Name: 0, dtype: float64

User features from W:


0   -0.074945
1   -0.278773
2   -0.389511
3    0.411489
4    0.337091
5    0.378174
6   -0.442824
7    0.329608
8    0.037669
9    0.221118
Name: 112, dtype: float64